# 第15课：OpenClaw 运行时初始化

本笔记本是课堂讲义。每个知识点包含：理论知识、案例代码、讲解、易错点与练习。综合练习 P1 使用课程根目录的教学运行时 [`agent_lab`](../agent_lab/README.md)。课后独立练习见 [chapter15_OpenClaw运行时_课后练习.ipynb](chapter15_OpenClaw运行时_课后练习.ipynb)。

**阶段定位**：阶段二 · 任务一 1.1 / 10分。10分

OpenClaw 在本课中的位置：把第 14 课的图和第 13 课的 /v1 收成**可握手的运行时**。评分只看一件事：Client 起来、探活过、首轮状态迁移日志能打印。

## 学习目标

1. 写出 OpenClawConfig：Base URL、Key、超时、model、session_id、agent_runtime。
2. 调用 handshake() 得到与阶段零一致的 HealthReport。
3. run_turn 一次并打印 state_log（turn / node / runtime）。
4. 说明 LangGraph 的 thread_id 如何映射为 session_id。

## 学习知识点

| 配置 | 握手 | 日志 |
| --- | --- | --- |
| base_url / api_key / timeout_s | probe 同源 | StateTransition |
| model / session_id | report.ok | turn, node, runtime |
| agent_runtime=openclaw | model_ids | 首轮必须非空 |

## 基础回顾与案例提问

1. **R.1** 只 `OpenClawClient()` 却不调用 handshake，算不算“初始化成功”？评分标准怎么写的？
2. **R.2** state_log 里的 runtime 字段从哪来？若拼错成 `open-claw` 会怎样？
3. **R.3** 握手用的端点与 run_turn 用的客户端是否应同源？为什么？

本课不展开 OpenClaw 商业控制台。教学实现见 `agent_lab.openclaw`。

使用 Python 3；需要 `pydantic`。从本课文件夹启动内核。本课不要求 GPU，也不强制安装 `langgraph` / `openai`。未配置私有化端点时，`get_client()` 返回进程内 Fake。不要使用 pandas。综合练习不要抄 `experiment.py` 的整段答案，按题面逐步完成。


In [ ]:
# R.1–R.3: Write and verify your predictions here.


In [ ]:
import sys
from pathlib import Path

COURSE = Path.cwd().resolve()
if COURSE.name.startswith("第"):
    COURSE = COURSE.parent
if str(COURSE) not in sys.path:
    sys.path.insert(0, str(COURSE))
print("已加入路径:", COURSE)
print("请从本课文件夹启动内核。未配置 OPENAI_BASE_URL 时使用教学 Fake 端点，不要求 GPU。")


## 1. 从状态图到运行时配置

### 理论知识

**图是算法，配置是部署。** session_id 对应 Checkpointer 的 thread_id；timeout_s 对应 Http 客户端；model 对应 /v1 的模型名。

### 案例：构造一份配置


In [ ]:
from agent_lab.openclaw import OpenClawConfig
cfg = OpenClawConfig(timeout_s=5.0, model="qwen2.5", session_id="handshake-1", agent_runtime="openclaw")
print(cfg.model_dump())


### 讲解

Field 上的 description 会进入后续 Prompt/文档。空的 base_url 表示使用阶段零的 Fake。

### 易错点与练习

1. **K1.1** 哪些项你会允许学生改，哪些应全班锁定？
2. **K1.2** agent_runtime 为什么要出现在日志里而不是只写在注释？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 2. Client 初始化

### 理论知识

**初始化 = 读配置 + 取客户端 + 编译图。** 此时还没有和模型握手。

### 案例：new 一个 Client


In [ ]:
from agent_lab.openclaw import OpenClawClient, OpenClawConfig
client = OpenClawClient(OpenClawConfig(timeout_s=5.0, session_id="handshake-1"))
print(type(client).__name__, client.source)


### 讲解

source 来自 get_client。课堂应为 fake。把它打印出来，避免有人以为已经连上了不存在的 GPU。

### 易错点与练习

1. **K2.1** Client 内部的 MemorySaver 与第 14 课的 saver 是不是同一个类？
2. **K2.2** 为什么图在 __init__ 里 compile，而不是每次 run_turn 重建？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 3. 握手探活

### 理论知识

**handshake 直接复用 probe。** 这是任务一 1.1 的硬指标：初始化成功还要探活。

### 案例：打印握手


In [ ]:
report = client.handshake()
print(report.ok, report.model_ids, report.source)


### 讲解

探活失败时不要继续聊天。评分老师会看这一行是否 ok=True。

### 易错点与练习

1. **K3.1** report 与第 13 课 HealthReport 是否同一模型？
2. **K3.2** 若真实端点超时，应改 Config.timeout_s 还是在节点里 sleep？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 4. 首轮状态迁移日志

### 理论知识

**run_turn 把 trace 翻译成 StateTransition 列表。** 字段：turn、node、message、runtime。

### 案例：跑一句健康探活


In [ ]:
result = client.run_turn("健康探活")
print(result["state_log"])
print(str(result.get("reply"))[:80])


### 讲解

教学图只有 agent 一个节点，所以日志很短。重要的是**有日志且 runtime 为 openclaw**，而不是日志越长越好。

### 易错点与练习

1. **K4.1** message 字段为什么重复了用户输入？便于哪一类对账？
2. **K4.2** 若 state_log 为空，优先检查哪两个调用？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 5. session 与多轮

### 理论知识

**默认 session_id 来自 Config。** 也可以在 run_turn(..., thread_id=) 覆盖，对应 LangGraph configurable.thread_id。

### 案例：看返回里的 thread_id


In [ ]:
print(result["thread_id"])


### 讲解

客服场景一个用户一个 session。不要把全班作业写进同一个默认 id 还声称互不影响。

### 易错点与练习

1. **K5.1** 两个同学都用 lab-session，Checkpointer 在同一进程里会怎样？
2. **K5.2** 什么时候应该显式传 thread_id？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 6. 超时与密钥

### 理论知识

**Key 在内网常是占位符，超时不是。** 把它们留在 Config，禁止写进 Prompt。

### 案例：读当前超时


In [ ]:
print("timeout_s", client.config.timeout_s)
print("api_key 长度", len(client.config.api_key))


### 讲解

打印密钥本身没有教学意义。作业若出现真实 Key，按泄密处理。

### 易错点与练习

1. **K6.1** 为什么探活超时和生成超时建议同一配置项？
2. **K6.2** Base URL 带不带尾斜杠应由谁 strip？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 7. 和阶段零对齐

### 理论知识

**同一 Fake，同一 qwen2.5。** OpenClaw 不是第二个模型供应商。

### 案例：对照模型名


In [ ]:
print(client.config.model)
print(report.model_ids)


### 讲解

若 Config.model 不在列表里，真实 vLLM 会 404。Fake 较宽松，但作业仍应保持一致。

### 易错点与练习

1. **K7.1** 列表有 deepseek-r1 时，为何本课默认仍用 qwen2.5？
2. **K7.2** 换模型名需要改几处配置？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 8. 10 分评分对照

### 理论知识

**得分点：Client 初始化成功、握手探活、打印首轮状态迁移日志。** 缺日志不得满分。

### 案例：自检三件事


In [ ]:
assert report.ok
assert result["state_log"]
assert result["state_log"][0]["runtime"] == "openclaw"
print("1.1 课堂自检通过")


### 讲解

experiment.py 就是这三件事的脚本版。讲义综合练习请自己敲，不要复制整文件。

### 易错点与练习

1. **K8.1** runtime 断言失败时，Config 哪一字段写错了？
2. **K8.2** 只打印 reply 不打印 state_log，会丢哪 10 分里的哪一部分？

**作答：** ____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 综合练习：握手 + 首轮日志

按 P1.1 → P1.2 → P1.3 顺序完成。每步都要能独立看出你做了什么。


### P1.1　构造 Client

写出 OpenClawConfig 与 OpenClawClient，打印 source。


In [ ]:
# P1.1: construct client.


### P1.2　handshake

打印 ok、model_ids、source，并确认 ok。


In [ ]:
# P1.2: handshake.


### P1.3　state_log

run_turn('健康探活')，打印 state_log，确认 runtime。


In [ ]:
# P1.3: first-turn state_log.


课后请打开 [chapter15_OpenClaw运行时_课后练习.ipynb](chapter15_OpenClaw运行时_课后练习.ipynb)。P1 对照 10 分清单，P2 解释配置映射，P3 选做两个 session。
